In [0]:
from pyspark.sql import functions as F
from datetime import date, timedelta
import requests

hoje = date.today()
dbutils.widgets.text("data_inicio", (hoje - timedelta(days=7)).strftime("%m-%d-%Y"))
dbutils.widgets.text("data_fim", hoje.strftime("%m-%d-%Y"))

BASE = "/Volumes/workspace/bronze/inputs/"

In [0]:
arquivos = {
    "movies_info_TMDB_IMDB.csv":       "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv":    "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv":  "bronze.tb_credits_and_tags",
    "movies_reviews.csv":              "bronze.tb_movies_reviews",
}

for arquivo, tabela in arquivos.items():
    df = (spark.read
          .option("header", True)
          .option("quote", '"')
          .option("escape", '"')
          .csv(BASE + arquivo))
    df = df.withColumn("ingestion_datetime", F.current_timestamp())
    df.write.format("delta").mode("append").saveAsTable(tabela)
    print(tabela, "->", df.count(), "linhas")

bronze.tb_movies_info -> 106930 linhas
bronze.tb_movies_financials -> 106165 linhas
bronze.tb_movies_metrics -> 107364 linhas
bronze.tb_credits_and_tags -> 106320 linhas
bronze.tb_movies_reviews -> 32412 linhas
